In [55]:
import os
import re
import bibtexparser


In [56]:

def create_bibtex_markdown(bibtex):

    # Parse BibTeX
    library = bibtexparser.loads(bibtex)
    entry = library.entries[0]

    # -----------------------------
    # Helper functions
    # -----------------------------

    def clean_tex(text):
        if not text:
            return ""

        text = re.sub(r"\\[a-zA-Z]+\{([^{}]*)\}", r"\1", text)

        replacements = {
            r"\'a": "á",
            r"\'e": "é",
            r"\'i": "í",
            r"\'o": "ó",
            r"\'u": "ú",
            r"\'A": "Á",
            r"\'E": "É",
            r"\'I": "Í",
            r"\'O": "Ó",
            r"\'U": "Ú",
            r"\'\i": "í",   # Bellog{\'\i}n
        }

        for k, v in replacements.items():
            text = text.replace(k, v)

        text = text.replace("{", "").replace("}", "")
        text = text.replace("\\&", "&")

        return text.strip() 


    def slugify(title):
        slug = title.lower()
        slug = re.sub(r"[^a-z0-9]+", "-", slug)
        slug = slug.strip("-")
        return slug


    # -----------------------------
    # Extract fields
    # -----------------------------

    title = clean_tex(entry.get("title", "Untitled"))
    authors = clean_tex(entry.get("author", ""))
    year = entry.get("year", "")
    month = entry.get("month", "1")
    day = entry.get("day", "1")
    venue = clean_tex(
        entry.get("booktitle", entry.get("journal", ""))
    )
    note = clean_tex(entry.get("note", ""))
    doi = entry.get("doi", "")

    url_slug = slugify(title)

    # GitHub Academic Pages expects YYYY-MM-DD
    pub_date = f"{year}-{int(month):02d}-{int(day):02d}"
    key = entry["ID"]   # "ungruh2026this"
    filename = f"{pub_date}-{key}.md"

    authors = clean_tex(entry.get("author", ""))

    # Remove any remaining LaTeX commands (e.g., \textbf, \emph, etc.)
    authors = re.sub(r"\\[a-zA-Z]+\{([^{}]*)\}", r"\1", authors)
    # Split authors
    author_list = authors.split(" and ")

    # Convert "Last, First" -> "First Last"
    formatted_authors = []
    for author in author_list:
        author = author.strip()
        if author.count(",") == 1:
            last, first = [x.strip() for x in author.split(",")]
            author = f"{first} {last}"
        formatted_authors.append(author)

    authors = ", ".join(formatted_authors)

    citation = f'{authors}. ({year}). "{title}." '

    if venue:
        citation += f'<i>{venue}</i>'

    if note:
        citation += f". {note}"

    if doi:
        citation += f". https://doi.org/{doi}"

    citation += "."
    citation = citation.replace('"', '&quot;')
    # -----------------------------
    # YAML
    # -----------------------------

    md = f"""---
    title: "{title}"
    collection: publications
    permalink: /publication/{pub_date}-{key}
    authors: "{authors}"
    date: {pub_date}
    venue: "{venue}"
    """

    if doi:
        md += f'paperurl: "https://doi.org/{doi}"\n'

    md += f'citation: "{citation}"\n'
    md += "---\n\n"

    if note:
        md += f"**{note}**\n\n"

    if doi:
        md += f"[Read paper](https://doi.org/{doi})\n\n"

    md += f"Recommended citation: {citation}"

    # -----------------------------
    # Write file
    # -----------------------------

    os.makedirs("../_publications", exist_ok=True)

    with open(f"../_publications/{filename}", "w", encoding="utf-8") as f:
        f.write(md)

    print(f"Created: {filename}")

In [57]:
# Paste ONE BibTeX entry here
single_bibtex = r"""
@inproceedings{ungruh2026this,
  title={This Is The World (Recommender Systems Made It For You): Capturing Risk Pathways for Children through a Pluralistic Multi-Risk Assessment of Recommenders},
  author={{\textbf{Robin Ungruh}} and Pera, Maria Soledad},
  booktitle={Proceedings of the 34th ACM Conference on User Modeling, Adaptation and Personalization (UMAP '26)},
  year={2026},
  doi={10.1145/3774935.3806148}
}
"""

create_bibtex_markdown(single_bibtex)

Created: 2026-01-01-ungruh2026this.md


In [58]:
multiple_bibtex = r"""

@inproceedings{ungruh2024putting,
  title={Putting Popularity Bias Mitigation to the Test: A User-Centric Evaluation in Music Recommenders},
  author={{\textbf{Robin Ungruh}} and Dinnissen, Karlijn and Volk, Anja and Pera, Maria Soledad and Hauptmann, Hanna},
  booktitle={Proceedings of the 18th ACM Conference on Recommender Systems},
  pages={169--178},
  year={2024},
  month={10},
  day={08},
  doi={10.1145/3640457.3688102}
}


@article{ungruh2024ah,
  title={Ah, that's the great puzzle: On the Quest of a Holistic Understanding of the Harms of Recommender Systems on Children},
  author={{\textbf{Robin Ungruh}} and Pera, Maria Soledad},
  journal={DCDW, co-located with ACM IDC '24},
  year={2024},
  month={5},
  day={03},
  note={(Position Paper)},
  doi = {arXiv:2405.02050}
}


@article{ungruh2025mirror,
  title={Mirror, Mirror: Exploring Stereotype Presence Among Top-N Recommendations That May Reach Children},
  author={{\textbf{Robin Ungruh}} and Al Nahadi, Murtadha and Pera, Maria Soledad},
  journal={ACM Transactions on Recommender Systems},
  year={2025},
  month={3},
  day={6},
  publisher={ACM New York, NY},
  doi={10.1145/3721987}
}

@inproceedings{ungruh2025impact,
  title={The Impact of Mainstream-Driven Algorithms on Recommendations for Children},
  author={{\textbf{Robin Ungruh}} and Bellog{\'\i}n, Alejandro and Pera, Maria Soledad},
  booktitle={European Conference on Information Retrieval},
  pages={67--84},
  year={2025},
  month={4},
  day={4},
  organization={Springer},
  doi={10.1007/978-3-031-88714-7_5}
}

@inproceedings{ungruh2025monolith,
  title={From Monolith to Mosaic: Uncovering Behavioral Differences for Choice Models in Recommender Systems Simulations},
  author={{\textbf{Robin Ungruh}} and Bellog{\'\i}n, Alejandro and Pera, Maria Soledad},
  booktitle={Proceedings of the 48th international ACM SIGIR conference on research and development in information retrieval},
  year={2025},
  month={7},
  day={13},
  doi={10.1145/3726302.3730199}
}




@InProceedings{heijne2025blurred,
  author="Heijne, Jasper
and {\textbf{Robin Ungruh}}
and Pera, Maria Soledad",
  title="Blurred Lines: Understanding the Fit of Song Lyrics in Music Catalogs That Can Reach Children Through Recommendations",
  booktitle="Advances in Bias, Fairness, and Understudied Users in Information Retrieval",
  year="2025",
  month={7},
  day={17},
  publisher="Springer Nature Switzerland",
  pages="60--75",
  doi={10.1007/978-3-032-12717-4_5}
}


@inproceedings{ungruh2025recommender,
  title={Are Recommender Systems Serving Children? Toward Child-Aware Design and Evaluation},
  author={{\textbf{Robin Ungruh}}},
  booktitle={Proceedings of the Nineteenth ACM Conference on Recommender Systems},
  pages={1451--1457},
  year={2025},
  month={9},
  day={22},
  doi={10.1145/3705328.3748752}
}

@inproceedings{ungruh2025previous,
  title={From Previous Plays to Long-Term Tastes: Exploring the Long-term Reliability of Recommender Systems Simulations for Children},
  author={{\textbf{Robin Ungruh}} and Bellog{\'\i}n, Alejandro and Pera, Maria Soledad},
  booktitle={Proceedings of the Nineteenth ACM Conference on Recommender Systems},
  pages={1193--1198},
  year={2025},
  month={9},
  day={22},
  doi={10.1145/3705328.3759301}
}

@inproceedings{ungruh2025impacts,
  title={Impacts of Mainstream-Driven Algorithms on Recommendations for Children Across Domains: A Reproducibility Study},
  author={{\textbf{Robin Ungruh}} and Bellog{\'\i}n, Alejandro and Kowald, Dominik and Pera, Maria Soledad},
  booktitle={Proceedings of the Nineteenth ACM Conference on Recommender Systems},
  pages={783--791},
  year={2025},
  month={9},
  day={22},
  doi={10.1145/3705328.3748160}
}

@article{ungruh2026flattening,
  title={Flattening Realities: Why Conventional Simulation Approaches Risk misrepresenting Children's Information Access},
  author={{\textbf{Robin Ungruh}} and Chakrabarti, Hrishita and Micol Tobia, Diletta and Bellog{\'\i}n, Alejandro and Landoni, Monica and Pera, Maria Soledad},
  journal={SynIRgy, co-located with ECIR '26},
  year={2026},
  month={4},
  day={2},
  doi={}
}

@inproceedings{ungruh2026this,
  title={This Is The World (Recommender Systems Made It For You): Capturing Risk Pathways for Children through a Pluralistic Multi-Risk Assessment of Recommenders},
  author={{\textbf{Robin Ungruh}} and Pera, Maria Soledad},
  booktitle={Proceedings of the 34th ACM Conference on User Modeling, Adaptation and Personalization (UMAP '26)},
  year={2026},
  month={6},
  day={8},
  doi={10.1145/3774935.3806148}
}

@inproceedings{ungruh2026preference,
  title={When Preference Is Not Enough: Why Recommender Systems Require Human-Aware Evaluation for Children},
  author={{\textbf{Robin Ungruh}} and Bellog{\'\i}n, Alejandro and Kowald, Dominik and Pera, Maria Soledad},
  booktitle={Proceedings of the 34th ACM Conference on User Modeling, Adaptation and Personalization (UMAP '26)},
  year={2026},
  doi={10.1145/3774935.3812719}
}

@inproceedings{ungruh2026some,
  title={Some People Are Worth Recommending For: Twenty Years of Progress, Gaps, and Opportunities For and With Children at RecSys},
  author={{\textbf{Robin Ungruh}} and Pera, Maria Soledad},
  booktitle={Proceedings of the Twentieth ACM Conference on Recommender Systems},
  year={2026},
  month={9},
  day={28},
  doi={10.1145/3773078.3831735}
}
"""


In [59]:
import re

entries = re.split(r'\n(?=@)', multiple_bibtex.strip())

for entry in entries:
    if entry.strip():
        create_bibtex_markdown(entry)

Created: 2024-10-08-ungruh2024putting.md
Created: 2024-05-03-ungruh2024ah.md
Created: 2025-03-06-ungruh2025mirror.md
Created: 2025-04-04-ungruh2025impact.md
Created: 2025-07-13-ungruh2025monolith.md
Created: 2025-07-17-heijne2025blurred.md
Created: 2025-09-22-ungruh2025recommender.md
Created: 2025-09-22-ungruh2025previous.md
Created: 2025-09-22-ungruh2025impacts.md
Created: 2026-04-02-ungruh2026flattening.md
Created: 2026-06-08-ungruh2026this.md
Created: 2026-01-01-ungruh2026preference.md
Created: 2026-09-28-ungruh2026some.md
